In [1]:
!pip install pandas numpy torch scikit-learn pyvi transformers datasets underthesea

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 110.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 80.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 79.5 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import torch
from pyvi import ViTokenizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from transformers import EarlyStoppingCallback
from underthesea import word_tokenize
from huggingface_hub import login

# Đăng nhập vào Hugging Face
login(token="YOUR_HF_TOKEN")

## 1. Tải dữ liệu và Tiền xử lý (Tách từ tiếng Việt)

In [3]:
def segment_text(text):
    return ViTokenizer.tokenize(str(text))

# Đọc dữ liệu
train_df = pd.read_csv('train.csv')
val_df = pd.read_csv('dev.csv')

# Gộp nhãn 1 và 2 thành 1
train_df['label_id'] = train_df['label_id'].replace(2, 1)
val_df['label_id'] = val_df['label_id'].replace(2, 1)

train_df['free_text'] = train_df['free_text'].apply(segment_text)
val_df['free_text'] = val_df['free_text'].apply(segment_text)

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

## 2. Tokenization với PhoBERT

In [4]:
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")

def tokenize_function(examples):
    return tokenizer(examples["free_text"], padding="max_length", truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)

# Đổi tên cột label_id thành labels để phù hợp với Trainer API
tokenized_train = tokenized_train.rename_column("label_id", "labels")
tokenized_val = tokenized_val.rename_column("label_id", "labels")

tokenized_train.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
tokenized_val.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/24048 [00:00<?, ? examples/s]

Map:   0%|          | 0/2672 [00:00<?, ? examples/s]

## 3. Cấu hình độ đo Macro F1 và Huấn luyện

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    macro_f1 = f1_score(labels, predictions, average='macro')
    return {"macro_f1": macro_f1}

# Cấu hình num_labels=2 sau khi gộp nhãn
model = AutoModelForSequenceClassification.from_pretrained("vinai/phobert-base-v2", num_labels=2)

training_args = TrainingArguments(
    output_dir="phobert-v2-vietnamese-sentiment",
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    push_to_hub=True,
    hub_strategy="every_save"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

trainer.train()
trainer.push_to_hub()
print("Đã tải mô hình lên Hugging Face Hub thành công!")

pytorch_model.bin:   0%|          | 0.00/540M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss


## 4. Dự đoán tập Test và Xuất file nộp bài

In [ ]:
# Đọc dữ liệu test
test_df = pd.read_csv('test.csv')
test_df['free_text'] = test_df['free_text'].apply(segment_text)
test_dataset = Dataset.from_pandas(test_df)
tokenized_test = test_dataset.map(tokenize_function, batched=True)
tokenized_test.set_format("torch", columns=["input_ids", "attention_mask"])

# Tiến hành dự đoán
predictions = trainer.predict(tokenized_test)
preds = np.argmax(predictions.predictions, axis=-1)

# Kiểm tra cột để tránh KeyError: 'id'
id_col = 'id' if 'id' in test_df.columns else test_df.columns[0]

submission = pd.DataFrame({
    'id': test_df[id_col],
    'label_id': preds
})
submission.to_csv('submission.csv', index=False)
print(f"Đã lưu tệp submission.csv thành công với cột ID là '{id_col}'!")